In [ ]:
!pip install numpy pandas matplotlib scikit-learn scipy seaborn

In [ ]:
import time
import numpy as np

from abc import ABC, abstractmethod

## Frank-Wolfe Algorithm
The Frank-Wolfe algorithm, also known as the Conditional Gradient Method, is an iterative optimization technique designed for constrained convex optimization problems. It works by iteratively solving linear approximations of the objective function over the feasible set.

### Step Size Strategies

The Frank-Wolfe algorithm supports several step size determination strategies:

1. **Exact Line Search**: Finds the optimal step size by solving
    γₖ = argmin_{γ∈[0,1]} f((1-γ)xₖ + γsₖ)
    This provides the best progress per iteration but requires solving a one-dimensional optimization problem. (Algorithm 3 FW.pdf and Section 4 FW_survey.pdf)

2. **Armijo Line Search**: A backtracking line search that ensures sufficient decrease in the objective. It starts with a large step size and reduces it until a sufficient decrease condition is met. (Section 4 FW_survey.pdf)

3. **Diminishing Step Size**: Uses a predefined schedule γₖ = 2/(k+2), which guarantees convergence while avoiding costly line searches. (Algorithm 1 FW.pdf, Section 4 FW_survey.pdf)

4. **Exact Line Search for Away and Pairwise Variants**: For away-step and pairwise variants, the exact line search is modified to:
   γₖ = argmin_{γ∈[0,γₘₐₓ]} f(x(ᵏ) + γdₖ)
   
   where γₘₐₓ may be restricted to less than 1 for away steps to ensure feasibility.

In [ ]:
class FrankWolfeBase(ABC):
    """Abstract base class for Frank-Wolfe algorithm variants"""

    def __init__(self, max_iter=1000, tol=1e-6, line_search_strategy='exact'):
        self.max_iter = max_iter
        self.tol = tol
        self.line_search_strategy = line_search_strategy

        self.iter = 0
        self.loss_history = []
        self.cpu_time_history = []
        self.gap_history = []

    @abstractmethod
    def fit(self, objective_function, compute_gradient, lmo, x0):
        """Solve the optimization problem starting from x0"""
        pass

    def _line_search(self, x, k, direction, grad, max_step_size=1.0):
        """Line search to find the optimal step size"""
        step_size = 0.0
        if self.line_search_strategy == 'exact':
            pass
        elif self.line_search_strategy == 'armijo':
            pass
        elif self.line_search_strategy == 'diminishing':
            step_size = 2 / (k + 2)
        elif self.line_search_strategy == 'exact_bounded':
            pass
        else:
            raise ValueError("Invalid line search strategy")

        return step_size

    def _compute_duality_gap(self, x, grad, s):
        """Compute the Frank-Wolfe duality gap"""
        return -grad @ (s - x)

### Frank Wolfe

*Algorithm 1 FW.pdf*

1. Start with an initial feasible point x₀
2. At each iteration k:
    - Compute the gradient ∇f(xₖ)
    - Find the point s in the feasible set that minimizes ⟨s, ∇f(xₖ)⟩
    - Determine the step size γₖ
    - Update: xₖ₊₁ = (1-γₖ)xₖ + γₖs

In [ ]:
class FrankWolfe(FrankWolfeBase):
    """Standard Frank-Wolfe algorithm implementation"""

    def __init__(self, max_iter=1000, tol=1e-6, line_search_strategy='exact'):
        super().__init__(max_iter, tol, line_search_strategy)

    def fit(self, objective_function, compute_gradient, lmo, x0):
        """Solve the optimization problem using standard Frank-Wolfe"""
        start_time = time.time()

        x = x0.copy()

        # Reset history
        self.loss_history = [objective_function(x)]
        self.cpu_time_history = [0.0]
        self.gap_history = []

        for k in range(self.max_iter):
            grad = compute_gradient(x)

            s = lmo(grad)

            gap = self._compute_duality_gap(x, grad, s)
            self.gap_history.append(gap)

            # Check convergence
            if gap < self.tol:
                break

            # Direction and step size
            d = s - x
            step_size = self._line_search(x, k, d, grad)

            # Update solution
            x = x + step_size * d

            # Update history
            self.iter = k + 1
            self.loss_history.append(objective_function(x))
            self.cpu_time_history.append(time.time() - start_time)

        return x

### Away Step Frank-Wolfe

*Algorithm 1 FW_variants.pdf*

1. Start with an initial feasible point x₀ and initialize active set S₀ = {x₀}
2. At each iteration k:
    - Compute the gradient ∇f(xₖ)
    - Find the Frank-Wolfe direction: Find sₖ in the feasible set that minimizes ⟨sₖ, ∇f(xₖ)⟩, set dᶠʷ = sₖ - xₖ
    - Find the Away direction: Find vₖ in the active set Sₖ that maximizes ⟨vₖ, ∇f(xₖ)⟩, set dᵃʷᵃʸ = xₖ - vₖ
    - Compute the Frank-Wolfe gap gᶠʷ = ⟨-∇f(xₖ), dᶠʷ⟩
    - If gᶠʷ ≤ ε, terminate (convergence)
    - Choose the direction with greater potential decrease:
      - If ⟨-∇f(xₖ), dᶠʷ⟩ ≥ ⟨-∇f(xₖ), dᵃʷᵃʸ⟩:
         - Use dₖ = dᶠʷ (Frank-Wolfe direction)
         - Set γₘₐₓ = 1
      - Else:
         - Use dₖ = dᵃʷᵃʸ (Away direction)
         - Set γₘₐₓ = αᵥₖ/(1-αᵥₖ) (maximum feasible step size)
    - Determine the step size γₖ through line search in [0, γₘₐₓ]
    - Update: xₖ₊₁ = xₖ + γₖdₖ
    - Update the active set Sₖ₊₁

In [ ]:
class AwayStepFrankWolfe(FrankWolfeBase):
    """Away Step Frank-Wolfe algorithm implementation"""

    def __init__(self, max_iter=1000, tol=1e-6):
        super().__init__(max_iter, tol, line_search_strategy='exact_bounded')

        self.active_set = None

    def _update_active_set(self, s, a_idx, step_size, max_step_size, is_away):
        """Update the active set

        If γt = γmax, then we call this step a drop step, as it fully removes the atom vt from the currently active set of atoms S(t) (by settings its weight to zero). The weight updates for lines 12 and 13 are of the following form: For a FW step, we have S(t+1) = {st} if γt = 1; otherwise S(t+1) = S(t) ∪{st}. Also, we have α(t+1) st := (1−γt)α(t) st +γt and α(t+1) v := (1−γt)α(t) v for v ∈ S(t) \{st}. For an away step, we have S(t+1) = S(t) \ {vt} if γt = γmax (a drop step); otherwise S(t+1) = S(t). Also, we have α(t+1) vt := (1 + γt)α(t) vt − γt and α(t+1) v := (1 + γt)α(t) v for v ∈ S(t) \ {vt}."""
        new_active_set = []

        # Away step case
        if is_away:
            is_drop_step = np.isclose(step_size, max_step_size)

            if is_drop_step:
                # Remove away vertex from active set
                for v_idx, (vertex, weight) in enumerate(self.active_set):
                    if v_idx != a_idx:
                        # Keep all vertices except the away vertex
                        new_weight = weight / (1 - self.active_set[self.a_idx][1])
                        if new_weight > 1e-10:  # Numerical stability check
                            new_active_set.append((vertex, new_weight))
            else:
                # Update weights
                for v_idx, (vertex, weight) in enumerate(self.active_set):
                    if v_idx == a_idx:
                        # For away vertex: α(t+1)_vt = (1+γt)α(t)_vt − γt
                        new_weight = (1 + step_size) * weight - step_size
                        if new_weight > 1e-10:  # Numerical stability check
                            new_active_set.append((vertex, new_weight))
                    else:
                        # For other vertices: α(t+1)_v = (1+γt)α(t)_v
                        new_active_set.append((vertex, (1 + step_size) * weight))
        # FW step case
        else:
            if np.isclose(step_size, 1.0):
                # Full step to s: S(t+1) = {st}
                new_active_set = [(s, 1.0)]
            else:
                # Partial step: S(t+1) = S(t) ∪ {st}
                s_in_active_set = False

                for vertex, weight in self.active_set:
                    new_weight = (1 - step_size) * weight

                    if np.array_equal(vertex, s):
                        # For FW vertex: α(t+1)_st = (1−γt)α(t)_st + γt
                        new_weight += step_size
                        s_in_active_set = True

                    if new_weight > 1e-10:  # Numerical stability check
                        new_active_set.append((vertex, new_weight))

                # If s not in active set, add it
                if not s_in_active_set:
                    new_active_set.append((s, step_size))

        return new_active_set

    def _compute_away_direction(self, grad):
        """Find the vertex in the active set with maximum inner product with gradient

        max ⟨-∇f(x), x - v_t⟩
        ⟨-∇f(x), x - v_t⟩ = ⟨-∇f(x), x⟩ - ⟨-∇f(x), v_t⟩ = -⟨∇f(x), x⟩ + ⟨∇f(x), v_t⟩, where -⟨∇f(x), x⟩ is constant"""
        max_inner_product = -float('inf')
        away_vertex = None
        away_idx = None

        for i, (atom, weight) in enumerate(self.active_set):
            inner_product = grad @ atom
            if inner_product > max_inner_product:
                max_inner_product = inner_product
                away_vertex = atom
                away_idx = i

        return away_vertex, away_idx

    def fit(self, objective_function, compute_gradient, lmo, x0):
        """Solve the optimization problem using Away Step Frank-Wolfe"""
        start_time = time.time()

        x = x0.copy()

        # Initialize active set: S(0) := {x(0)} with α(0)_x(0) = 1
        self.active_set = [(x0, 1.0)]

        # Reset history
        self.loss_history = [objective_function(x)]
        self.cpu_time_history = [0.0]
        self.gap_history = []

        for k in range(self.max_iter):
            grad = compute_gradient(x)

            # Compute FW direction
            s = lmo(grad)
            d_fw = s - x
            gap_fw = -(grad @ d_fw)

            # Check convergence
            if gap_fw < self.tol:
                break

            # Compute away direction
            a, a_idx = self._compute_away_direction(grad)
            d_away = x - a
            gap_away = -(grad @ d_away)
            is_away = (gap_away > gap_fw)

            # Choose direction based on gap
            d = None
            max_step_size = None
            if is_away:
                d = d_away
                away_weight = self.active_set[a_idx][1]
                max_step_size = away_weight / (1.0 - away_weight)
            else:
                d = d_fw
                max_step_size = 1.0

            # Step size
            step_size = self._line_search(x, k, d, grad, max_step_size)

            # Update solution
            x = x + step_size * d

            # Update active set
            self.active_set = self._update_active_set(s, a_idx, step_size, max_step_size, is_away)

            # Update history
            self.iter = k + 1
            self.gap_history.append(gap_away if is_away else gap_fw)
            self.loss_history.append(objective_function(x))
            self.cpu_time_history.append(time.time() - start_time)

        return x

### Pairwise Frank-Wolfe



## Test on Convex Quadratic Function

## Test on Max-Clique Problem